# 🎬 AI Video Studio — Free Colab GPU Worker

This notebook turns Google Colab's **free T4 GPU** into a generation backend for your local **AI Video Studio** (`app.py`).

**How it works:** this notebook loads **Wan2.1-T2V-1.3B** on the Colab GPU and starts a tiny Gradio server with a public `*.gradio.live` URL. You paste that URL into the studio's **Advanced → Backends** panel, and every video you generate locally is rendered on Colab's GPU — at full model quality, for free.

### Steps
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run every cell top to bottom (`Runtime → Run all`).
3. Copy the `https://xxxxx.gradio.live` URL printed by the **last** cell.
4. In the local studio: **Advanced → 🔌 Backends & credentials → Colab worker URL** → paste → **Save & refresh backends**.
5. Keep this Colab tab open while you generate. Free sessions idle-out after a while — just re-run if the URL dies.

> First run downloads the model weights (~17 GB) to the Colab session; this takes a few minutes and is normal.

## 1. Verify the GPU

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0),
      '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Install dependencies + fetch the Wan2.1 code

In [ ]:
%cd /content
![ -d Wan2.1 ] || git clone https://github.com/Wan-Video/Wan2.1.git
%cd /content/Wan2.1
!pip install -q -r requirements.txt
!pip install -q "gradio==4.44.1" "huggingface_hub[cli]" ftfy dashscope
print('deps installed')

## 3. Download the T2V-1.3B weights (once per session)

In [ ]:
from huggingface_hub import snapshot_download
CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
snapshot_download('Wan-AI/Wan2.1-T2V-1.3B', local_dir=CKPT_DIR,
                  resume_download=True)
print('weights ready at', CKPT_DIR)

## 4. Load the model once
Cached in memory so every request is fast.

In [ ]:
import sys, torch
sys.path.insert(0, '/content/Wan2.1')
import wan
from wan.configs import WAN_CONFIGS

CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
cfg = WAN_CONFIGS['t2v-1.3B']
print('Loading Wan2.1-T2V-1.3B (offloaded to fit T4 16GB)...')
MODEL = wan.WanT2V(
    config=cfg, checkpoint_dir=CKPT_DIR, device_id=0,
    rank=0, t5_fsdp=False, dit_fsdp=False, use_usp=False,
    t5_cpu=True,          # keep the text encoder on CPU -> fits the 16GB T4
)
print('model loaded')

## 5. Start the worker server
Exposes `/generate` with the exact signature the local `ColabBackend` calls:
`(prompt, negative_prompt, width, height, num_frames, steps, guidance, seed)`.

In [ ]:
import gradio as gr
from wan.utils.utils import cache_video
import tempfile, torch

def generate(prompt, negative_prompt, width, height, num_frames,
             steps, guidance, seed):
    width, height = int(width), int(height)
    num_frames = int(num_frames)
    if (num_frames - 1) % 4 != 0:            # Wan needs 4n+1 frames
        num_frames = (num_frames // 4) * 4 + 1
    seed = int(seed)
    if seed < 0:
        seed = torch.randint(0, 2**31 - 1, (1,)).item()
    print(f'gen {width}x{height} f{num_frames} s{steps} seed{seed}: {prompt[:60]}')
    video = MODEL.generate(
        prompt,
        size=(width, height),
        frame_num=num_frames,
        sampling_steps=int(steps),
        guide_scale=float(guidance),
        n_prompt=negative_prompt or '',
        seed=seed,
        offload_model=True,                  # frees VRAM between stages on T4
    )
    out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    cache_video(video[None], save_file=out, fps=16,
                nrow=1, normalize=True, value_range=(-1, 1))
    return out

with gr.Blocks(title='Wan2.1 Colab Worker') as worker:
    gr.Markdown('### Wan2.1 T2V-1.3B worker — leave this tab open')
    with gr.Row():
        p  = gr.Textbox(label='prompt')
        np_ = gr.Textbox(label='negative_prompt')
    with gr.Row():
        w = gr.Number(832, label='width');  h = gr.Number(480, label='height')
        nf = gr.Number(81, label='num_frames'); st = gr.Number(30, label='steps')
        g = gr.Number(6.0, label='guidance'); sd = gr.Number(-1, label='seed')
    vid = gr.Video(label='result')
    gr.Button('Generate', variant='primary').click(
        generate, [p, np_, w, h, nf, st, g, sd], vid, api_name='generate')

worker.queue(max_size=20)
print('\n' + '='*70)
print('COPY THE  https://xxxxx.gradio.live  URL BELOW into the local studio:')
print('='*70)
worker.launch(share=True)